# Predicting subsurface eddy tilt from surface signatures

This notebook tests whether surface properties contain predictive information about subsurface eddy tilt. It is a feasibility and generalisation experiment, not an attempt to fill unknown observations.

The primary safeguards are:

- predict eastward and northward tilt components rather than regressing directly on a circular bearing;
- keep all observations from an eddy in the same data partition;
- compare nonlinear and linear models with a mean-vector baseline;
- evaluate displacement-vector, distance, and circular-direction errors separately;
- calculate feature importance only on eddies excluded from fitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt
import ml_subsurface_tools as ml

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42

## 1. Load the surface and tilt data

`add_pv_gradient_terms(..., core_mean=True)` calculates the shallow-water PV-gradient terms using core-mean bathymetry and bathymetric gradients. The helper calls its total gradient magnitude `PV_grad_mag`; here it is copied to the requested modelling name `PV_gradient`.

In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)

df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
df_eddies["PV_gradient"] = df_eddies["PV_grad_mag"]

print(f"Rows: {len(df_eddies):,}")
print(f"Unique eddies: {df_eddies['Eddy'].nunique():,}")
df_eddies[ml.FEATURES + ["TiltDis", "TiltDir"]].head()

### Target-definition checkpoint

Before interpreting model skill, confirm that `TiltDis` and `TiltDir` describe tilt over the same vertical interval for every eddy-day. If the terminal depth varies, absolute distance also represents vertical-profile length. In that case, repeat the analysis with tilt to a fixed depth or with tilt distance divided by vertical separation.

## 2. Audit target availability and prepare the modelling table

Missing tilt values may not be random. Coverage is reported overall and by polarity before complete cases are selected. Conclusions from the model apply to eddy-days for which a valid tilt was calculated.

In [ ]:
coverage = ml.target_availability_summary(df_eddies)
coverage

In [ ]:
model_df = ml.prepare_modelling_table(df_eddies)

print(f"Complete modelling rows: {len(model_df):,}")
print(f"Independent eddy groups: {model_df['Eddy'].nunique():,}")
print(f"Retained fraction: {len(model_df) / len(df_eddies):.1%}")
model_df[ml.FEATURES + ["TiltDis", "TiltDir", "TiltEast", "TiltNorth"]].describe(include="all").T

`TiltDir` is a compass bearing (0° north, 90° east). The targets are therefore

```text
TiltEast  = TiltDis × sin(TiltDir)
TiltNorth = TiltDis × cos(TiltDir)
```

Distance and bearing are reconstructed after prediction. This makes 359° and 1° neighbouring directions rather than opposite numerical extremes.

In [ ]:
model_df[ml.NUMERIC_FEATURES + ["TiltDis"]].hist(bins=50, figsize=(13, 10))
plt.suptitle("Predictor and target distributions", y=1.01)
plt.tight_layout()

## 3. Hold out complete eddies

A random row split would leak adjacent days from the same eddy into training and testing. `GroupShuffleSplit` instead holds out 20% of eddy identities. The test set is untouched during fitting and is used for the final feasibility assessment.

In [ ]:
split = ml.grouped_train_test_split(
    model_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

assert set(split.groups_train).isdisjoint(set(split.groups_test))
pd.Series({
    "training rows": len(split.X_train),
    "test rows": len(split.X_test),
    "training eddies": split.groups_train.nunique(),
    "test eddies": split.groups_test.nunique(),
})

## 4. Fit baseline, linear, and nonlinear models

The predictors are exactly:

```python
['lat', 'w', 'Omega', 'Rc', 'AR', 'Age', 'PV_gradient']
```

with `Cyc` treated as a categorical predictor. `Rc` is used rather than `R`. Ridge regression tests approximately linear signal; histogram gradient boosting captures nonlinearities and interactions. Both are compared with the training-set mean tilt vector.

In [ ]:
models = ml.build_models(random_state=RANDOM_STATE)
fitted_models = ml.fit_models(models, split)
scores, predictions = ml.evaluate_models(fitted_models, split)
scores.round(3)

In [ ]:
ml.plot_model_comparison(scores);

The primary metric is mean displacement-vector error because it jointly penalises distance and direction. Distance MAE measures magnitude skill. Angular error is reported separately, but should not dominate interpretation when observed tilt is close to zero and its direction is physically unstable. A useful model must improve materially on the mean-vector baseline on unseen eddies.

## 5. Diagnose the best predictive model

In [ ]:
candidate_scores = scores.drop(index="Mean-vector baseline", errors="ignore")
best_name = candidate_scores["vector_MAE_km"].idxmin()
best_prediction = predictions[best_name]
print(f"Best candidate on held-out eddies: {best_name}")
ml.plot_prediction_diagnostics(best_prediction, title=best_name);

Direction is ill-defined for very small displacement. The next table repeats circular metrics after imposing progressively larger observed-tilt thresholds.

In [ ]:
ml.direction_performance_by_tilt(best_prediction, thresholds=(0, 5, 10, 20)).round(3)

Performance is also separated by polarity. A large AE/CE contrast may justify fitting separate models later, but the first experiment keeps all data together and includes `Cyc` categorically.

In [ ]:
ml.performance_by_group(
    best_prediction,
    model_df.loc[split.test_index],
    group_column="Cyc",
).round(3)

## 6. Grouped cross-validation

The single held-out split gives a clear final test. Five-fold grouped cross-validation measures sensitivity to which eddies are held out. This cell can take several minutes because both models are refitted five times.

In [ ]:
cv_results = ml.grouped_cross_validation(models, model_df, n_splits=5)
cv_summary = cv_results.groupby("model").agg(
    vector_MAE_mean=("vector_MAE_km", "mean"),
    vector_MAE_std=("vector_MAE_km", "std"),
    distance_MAE_mean=("distance_MAE_km", "mean"),
    angular_error_median=("median_angular_error_deg", "median"),
    within_30deg_mean=("within_30deg_fraction", "mean"),
)
cv_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, part in cv_results.groupby("model"):
    ax.plot(part["fold"], part["vector_MAE_km"], marker="o", label=name)
ax.set(xlabel="Grouped fold", ylabel="Mean tilt-vector error (km)", xticks=range(1, 6))
ax.set_title("Generalisation to unseen eddies across folds")
ax.legend(frameon=False)
plt.tight_layout()

## 7. Held-out permutation importance

Each raw predictor is shuffled in the untouched test set. Importance is the resulting increase in mean vector error. This measures predictive reliance, not causality; correlated predictors can share or mask importance.

In [ ]:
importance = ml.raw_feature_permutation_importance(
    fitted_models[best_name],
    split.X_test,
    split.y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
)
display(importance.round(4))
ml.plot_permutation_importance(importance);

## 8. Interpretation checklist and next experiments

Treat surface-to-tilt prediction as feasible only when the nonlinear and/or linear model improves consistently over the baseline in both the held-out split and grouped folds. Report vector error first, followed by distance and thresholded angular metrics.

Before drawing a physical conclusion:

1. verify that tilt uses a common reference depth;
2. quantify how missing tilt depends on polarity and surface predictors;
3. inspect whether performance differs materially between AE and CE;
4. repeat with entire years held out to test temporal transfer;
5. compare this surface-only feature set with a context model containing season, longitude, region, or bathymetry;
6. do not interpret permutation importance as causal evidence.

A useful scientific result can be either positive or negative: strong out-of-eddy skill indicates predictable surface information, while weak skill after leakage-safe validation shows that these surface signatures alone do not constrain instantaneous subsurface tilt sufficiently.